# Giao diện Kiểm định và Dán nhãn Dữ liệu Âm thanh (TTS Dataset Annotator UI)
Sử dụng công cụ này trên môi trường Jupyter Notebook hoặc Google Colab để duyệt nhanh danh sách file audio thô, chỉnh sửa văn bản, và phân loại dữ liệu sạch / dữ liệu ồn / dữ liệu lỗi.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q ipywidgets pandas pydub

In [ ]:
import os
import shutil
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

# ================= CONFIGURATION =================
METADATA_CSV = "processed_data/raw_sliced_metadata.csv" # File CSV gốc sau khi chạy slicer.py
AUDIO_DIR = "processed_data/wavs"                      # Thư mục chứa audio segment

VERIFIED_CSV = "processed_data/metadata_verified.csv"  # Lưu file sạch
NOISE_CSV = "processed_data/metadata_noise.csv"        # Lưu file ồn chờ xử lý
TRASH_DIR = "processed_data/wavs_trash"                # Chứa file lỗi
NOISE_DIR = "processed_data/wavs_need_denoise"         # Chứa file ồn để chạy DeepFilterNet
# =================================================

# Tạo các thư mục nếu chưa tồn tại
os.makedirs(TRASH_DIR, exist_ok=True)
os.makedirs(NOISE_DIR, exist_ok=True)

class DatasetAnnotator:
    def __init__(self):
        if not os.path.exists(METADATA_CSV):
            print(f"[-] Lỗi: Không tìm thấy file metadata tại {METADATA_CSV}. Vui lòng chạy slicer.py trước.")
            return
            
        self.df_meta = pd.read_csv(METADATA_CSV)
        self.total_items = len(self.df_meta)
        self.current_index = 0
        
        # Nạp tiến trình cũ nếu có (bỏ qua những file đã xử lý)
        self.processed_files = set()
        if os.path.exists(VERIFIED_CSV):
            df_v = pd.read_csv(VERIFIED_CSV)
            self.processed_files.update(df_v['audio_path'].tolist())
        if os.path.exists(NOISE_CSV):
            df_n = pd.read_csv(NOISE_CSV)
            self.processed_files.update(df_n['audio_path'].tolist())
            
        # Tìm vị trí chỉ số tiếp theo chưa xử lý
        self.find_next_unannotated()
        
        # Biến lưu trữ hành động cuối để Hoàn Tác (Undo)
        self.history = []
        
        # Tạo các widgets giao diện
        self.setup_ui()
        self.render_current()

    def find_next_unannotated(self):
        while self.current_index < self.total_items:
            audio_path = self.df_meta.iloc[self.current_index]['audio_path']
            if audio_path not in self.processed_files:
                break
            self.current_index += 1
            
    def setup_ui(self):
        self.title_label = widgets.HTML("<h2><b>TTS Dataset Annotator UI</b></h2>")
        self.progress_label = widgets.Label(value="")
        self.audio_widget = widgets.Output()
        self.text_input = widgets.Text(value="", layout=widgets.Layout(width='90%'), description='Văn bản:')
        self.path_label = widgets.Label(value="")
        
        # Nút hành động
        self.btn_verified = widgets.Button(description="Chuẩn / Lưu", button_style='success', icon='check')
        self.btn_noise = widgets.Button(description="Ồn / Nhạc", button_style='warning', icon='volume-up')
        self.btn_trash = widgets.Button(description="Rác (Lỗi)", button_style='danger', icon='trash')
        self.btn_undo = widgets.Button(description="Hoàn tác (Undo)", button_style='info', icon='undo')
        
        # Liên kết sự kiện click
        self.btn_verified.on_click(self.on_verified)
        self.btn_noise.on_click(self.on_noise)
        self.btn_trash.on_click(self.on_trash)
        self.btn_undo.on_click(self.on_undo)
        
        self.layout_buttons = widgets.HBox([self.btn_verified, self.btn_noise, self.btn_trash, self.btn_undo])
        
        # Giao diện chính
        self.main_container = widgets.VBox([
            self.title_label,
            self.progress_label,
            self.path_label,
            self.audio_widget,
            self.text_input,
            self.layout_buttons
        ])
        display(self.main_container)
        
    def render_current(self):
        if self.current_index >= self.total_items:
            self.progress_label.value = "[+] Đã hoàn thành duyệt toàn bộ bộ dữ liệu!"
            self.text_input.value = ""
            self.path_label.value = ""
            with self.audio_widget:
                clear_output()
            return
            
        row = self.df_meta.iloc[self.current_index]
        audio_rel_path = row['audio_path']
        # Đường dẫn thực tế trên môi trường chạy
        audio_real_path = os.path.join(os.path.dirname(METADATA_CSV), "wavs", os.path.basename(audio_rel_path))
        
        self.progress_label.value = f"Tiến độ: {self.current_index + 1} / {self.total_items} (Đã duyệt {len(self.processed_files)} file)"
        self.path_label.value = f"Đang xử lý file: {audio_real_path}"
        self.text_input.value = str(row['text'])
        
        # Tạo trình phát nhạc
        with self.audio_widget:
            clear_output()
            if os.path.exists(audio_real_path):
                display(Audio(audio_real_path, autoplay=True))
            else:
                print(f"[!] Lỗi: Không tìm thấy file âm thanh tại: {audio_real_path}")

    def save_log(self, csv_file, audio_path, text):
        header = not os.path.exists(csv_file)
        df = pd.DataFrame([{"audio_path": audio_path, "text": text}])
        df.to_csv(csv_file, mode='a', index=False, header=header, encoding='utf-8')
        
    def on_verified(self, b):
        row = self.df_meta.iloc[self.current_index]
        audio_path = row['audio_path']
        text = self.text_input.value.strip()
        
        self.save_log(VERIFIED_CSV, audio_path, text)
        self.processed_files.add(audio_path)
        self.history.append({'action': 'verified', 'audio_path': audio_path, 'text': text})
        
        self.current_index += 1
        self.find_next_unannotated()
        self.render_current()
        
    def on_noise(self, b):
        row = self.df_meta.iloc[self.current_index]
        audio_path = row['audio_path']
        text = self.text_input.value.strip()
        
        # Di chuyển file audio sang thư mục Ồn để tiền xử lý
        src_path = os.path.join(os.path.dirname(METADATA_CSV), "wavs", os.path.basename(audio_path))
        dest_path = os.path.join(NOISE_DIR, os.path.basename(audio_path))
        
        if os.path.exists(src_path):
            shutil.move(src_path, dest_path)
            
        self.save_log(NOISE_CSV, audio_path, text)
        self.processed_files.add(audio_path)
        self.history.append({'action': 'noise', 'audio_path': audio_path, 'text': text, 'src': src_path, 'dest': dest_path})
        
        self.current_index += 1
        self.find_next_unannotated()
        self.render_current()
        
    def on_trash(self, b):
        row = self.df_meta.iloc[self.current_index]
        audio_path = row['audio_path']
        
        # Di chuyển file audio sang Thùng rác
        src_path = os.path.join(os.path.dirname(METADATA_CSV), "wavs", os.path.basename(audio_path))
        dest_path = os.path.join(TRASH_DIR, os.path.basename(audio_path))
        
        if os.path.exists(src_path):
            shutil.move(src_path, dest_path)
            
        self.processed_files.add(audio_path)
        self.history.append({'action': 'trash', 'audio_path': audio_path, 'src': src_path, 'dest': dest_path})
        
        self.current_index += 1
        self.find_next_unannotated()
        self.render_current()
        
    def on_undo(self, b):
        if not self.history:
            print("[!] Không có hành động nào để hoàn tác.")
            return
            
        last_action = self.history.pop()
        action_type = last_action['action']
        audio_path = last_action['audio_path']
        
        # 1. Khôi phục file audio nếu bị di chuyển
        if action_type in ['noise', 'trash']:
            src = last_action['src']
            dest = last_action['dest']
            if os.path.exists(dest):
                shutil.move(dest, src)
                
        # 2. Xóa dòng cuối trong log CSV tương ứng
        target_csv = VERIFIED_CSV if action_type == 'verified' else (NOISE_CSV if action_type == 'noise' else None)
        if target_csv and os.path.exists(target_csv):
            df = pd.read_csv(target_csv)
            if not df.empty:
                df = df[:-1] # Cắt bỏ dòng cuối cùng
                if df.empty:
                    os.remove(target_csv)
                else:
                    df.to_csv(target_csv, index=False, encoding='utf-8')
                    
        # 3. Cập nhật trạng thái bộ nhớ
        self.processed_files.remove(audio_path)
        
        # Quay lại phần tử cũ
        self.current_index = self.df_meta[self.df_meta['audio_path'] == audio_path].index[0]
        self.render_current()
        print(f"[+] Hoàn tác thành công hành động trước đó cho file: {os.path.basename(audio_path)}")

# Chạy widget
annotator = DatasetAnnotator()